# 衔接BND_convert.ipynb，对BND转换后的结果进行统计。

## 一、用到的脚本

### summarize_80_pairs_sv_by_tool.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import gzip
import re
from collections import Counter
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
BND_RE = re.compile(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]")


def open_text(path):
    path = str(path)
    if path.endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")


def parse_info(info):
    out = {}
    if not info or info == ".":
        return out
    for item in info.split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" in item:
            key, value = item.split("=", 1)
            out[key.strip()] = value.strip()
        else:
            out[item] = True
    return out


def norm_chrom(chrom):
    chrom = str(chrom or "").strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    chrom = chrom.upper()
    if chrom == "M":
        chrom = "MT"
    return chrom


def chrom_sort_key(chrom):
    chrom = norm_chrom(chrom)
    if chrom.isdigit():
        return (0, int(chrom))
    if chrom == "X":
        return (0, 23)
    if chrom == "Y":
        return (0, 24)
    if chrom == "MT":
        return (0, 25)
    return (1, chrom)


def to_int(value, default=None):
    try:
        if value in {None, "", ".", "NA"}:
            return default
        return int(float(str(value).split(",")[0]))
    except Exception:
        return default


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    mapping = {
        "DELETION": "DEL",
        "DUPLICATION": "DUP",
        "INVERSION": "INV",
        "INSERTION": "INS",
        "TRANSLOCATION": "TRA",
        "CTX": "TRA",
        "BREAKEND": "BND",
    }
    return mapping.get(sv, sv)


def extract_svtype(info_text, alt, info=None):
    if info is None:
        info = {}

    for key in ("SVTYPE", "svtype", "SvType"):
        if key in info:
            sv = norm_svtype(info.get(key))
            if sv != "UNKNOWN":
                return sv

    match = re.search(r"(?i)(?:^|;)SVTYPE=([^;\t\r\n ]+)", info_text or "")
    if match:
        sv = norm_svtype(match.group(1))
        if sv != "UNKNOWN":
            return sv

    alt_text = str(alt or "").strip().upper()
    match = re.search(r"<([^<>:,]+)>", alt_text)
    if match:
        sv = norm_svtype(match.group(1))
        if sv != "UNKNOWN":
            return sv

    if "[" in alt_text or "]" in alt_text:
        return "BND"

    return "UNKNOWN"


def bnd_remote(alt):
    match = BND_RE.search(alt or "")
    if not match:
        return None, None
    return norm_chrom(match.group(1)), int(match.group(2))


def length_bin(length, is_trans):
    if is_trans:
        return "TRA_or_interchrom"
    if length is None:
        return "unknown"
    if length < 50:
        return "<50 bp"
    if length < 100:
        return "50-100 bp"
    if length < 1000:
        return "100 bp-1 kb"
    if length < 10000:
        return "1-10 kb"
    if length < 100000:
        return "10-100 kb"
    if length < 1000000:
        return "100 kb-1 Mb"
    return ">=1 Mb"


def get_manifest_rows(manifest):
    with open(manifest, "rt") as handle:
        return list(csv.DictReader(handle, delimiter="\t"))


def parse_vcf(vcf, pair_id, tumor_id, normal_id, tool, pass_only=True):
    records = []
    if not vcf or vcf == "NA" or not Path(vcf).exists():
        return records

    with open_text(vcf) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue

            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue

            chrom, pos, rec_id, ref, alt, qual, filt, info_text = fields[:8]
            if pass_only and filt not in {"PASS", "."}:
                continue

            info = parse_info(info_text)
            svtype = extract_svtype(info_text, alt, info)
            chrom1 = norm_chrom(chrom)
            pos1 = to_int(pos)
            if pos1 is None:
                continue

            chrom2 = norm_chrom(info.get("CHR2") or info.get("CHROM2") or info.get("ENDCHR") or chrom1)
            pos2 = to_int(info.get("END") or info.get("POS2") or info.get("ENDPOS"))

            if svtype in {"BND", "TRA"}:
                remote_chrom, remote_pos = bnd_remote(alt)
                if remote_chrom:
                    chrom2 = remote_chrom
                if remote_pos:
                    pos2 = remote_pos

            svlen = to_int(info.get("SVLEN"))
            if pos2 is None:
                if svlen not in {None, 0}:
                    pos2 = pos1 + abs(svlen)
                else:
                    pos2 = pos1

            is_trans = chrom1 != chrom2 or svtype == "TRA"
            if svlen not in {None, 0}:
                abs_len = abs(svlen)
            elif is_trans:
                abs_len = None
            else:
                abs_len = abs(pos2 - pos1)

            start = min(pos1, pos2) if not is_trans else pos1
            end = max(pos1, pos2) if not is_trans else pos2

            records.append({
                "pair_id": pair_id,
                "tumor_id": tumor_id,
                "normal_id": normal_id,
                "tool": tool,
                "vcf": vcf,
                "record_id": rec_id or f"{chrom1}:{pos1}:{svtype}",
                "chrom1": chrom1,
                "pos1": pos1,
                "chrom2": chrom2,
                "pos2": pos2,
                "start": start,
                "end": end,
                "svtype": svtype,
                "svlen": abs_len if abs_len is not None else "NA",
                "length_bin": length_bin(abs_len, is_trans),
                "relation": "trans" if is_trans else "cis",
                "filter": filt,
            })

    return records


def write_table(path, rows, columns):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wt", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns, delimiter="\t", extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def count_rows(records, keys, count_name="count"):
    counter = Counter(tuple(record[k] for k in keys) for record in records)
    rows = []
    for key_tuple, count in sorted(counter.items()):
        row = {k: v for k, v in zip(keys, key_tuple)}
        row[count_name] = count
        rows.append(row)
    return rows


def chrom_distribution(records):
    counter = Counter()
    for rec in records:
        chroms = {rec["chrom1"], rec["chrom2"]}
        for chrom in chroms:
            counter[(rec["pair_id"], rec["tumor_id"], rec["normal_id"], rec["tool"], chrom)] += 1

    rows = []
    for (pair_id, tumor_id, normal_id, tool, chrom), count in sorted(
        counter.items(),
        key=lambda x: (x[0][0], x[0][3], chrom_sort_key(x[0][4])),
    ):
        rows.append({
            "pair_id": pair_id,
            "tumor_id": tumor_id,
            "normal_id": normal_id,
            "tool": tool,
            "chrom": chrom,
            "sv_event_involvement_count": count,
        })
    return rows


def trans_chr_pair_distribution(records):
    counter = Counter()
    for rec in records:
        if rec["relation"] != "trans":
            continue
        chrom1, chrom2 = sorted([rec["chrom1"], rec["chrom2"]], key=chrom_sort_key)
        counter[(rec["pair_id"], rec["tumor_id"], rec["normal_id"], rec["tool"], chrom1, chrom2, rec["svtype"])] += 1

    rows = []
    for (pair_id, tumor_id, normal_id, tool, chrom1, chrom2, svtype), count in sorted(
        counter.items(),
        key=lambda x: (x[0][0], x[0][3], chrom_sort_key(x[0][4]), chrom_sort_key(x[0][5]), x[0][6]),
    ):
        rows.append({
            "pair_id": pair_id,
            "tumor_id": tumor_id,
            "normal_id": normal_id,
            "tool": tool,
            "chrom1": chrom1,
            "chrom2": chrom2,
            "svtype": svtype,
            "count": count,
        })
    return rows


def main():
    parser = argparse.ArgumentParser(description="Summarize PASS SV records by pair and tool. Tables only; no plotting.")
    parser.add_argument("--manifest", default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv")
    parser.add_argument("--outdir", default="/mnt/home/ygjx/chenkejin/80_sv_summary")
    parser.add_argument("--include-filtered", action="store_true", help="Include non-PASS records. Default uses PASS or . only.")
    args = parser.parse_args()

    outdir = Path(args.outdir)
    tables_dir = outdir / "tables"
    tables_dir.mkdir(parents=True, exist_ok=True)

    manifest_rows = get_manifest_rows(args.manifest)
    all_records = []

    for row in manifest_rows:
        pair_id = row["pair_id"]
        tumor_id = row["tumor_id"]
        normal_id = row["normal_id"]
        for tool in TOOLS:
            vcf = row.get(f"{tool}_vcf", "NA")
            records = parse_vcf(
                vcf=vcf,
                pair_id=pair_id,
                tumor_id=tumor_id,
                normal_id=normal_id,
                tool=tool,
                pass_only=not args.include_filtered,
            )
            all_records.extend(records)

    record_cols = [
        "pair_id", "tumor_id", "normal_id", "tool", "record_id",
        "chrom1", "pos1", "chrom2", "pos2", "start", "end",
        "svtype", "svlen", "length_bin", "relation", "filter", "vcf",
    ]

    write_table(tables_dir / "all_pass_sv_records.normalized.tsv", all_records, record_cols)
    write_table(
        tables_dir / "total_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool"], "total_sv_count"),
        ["pair_id", "tumor_id", "normal_id", "tool", "total_sv_count"],
    )
    write_table(
        tables_dir / "svtype_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "svtype"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "svtype", "count"],
    )
    write_table(
        tables_dir / "length_bin_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "length_bin"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "length_bin", "count"],
    )
    write_table(
        tables_dir / "cis_trans_counts.by_pair_tool.tsv",
        count_rows(all_records, ["pair_id", "tumor_id", "normal_id", "tool", "relation"]),
        ["pair_id", "tumor_id", "normal_id", "tool", "relation", "count"],
    )
    write_table(
        tables_dir / "chromosome_distribution.by_pair_tool.tsv",
        chrom_distribution(all_records),
        ["pair_id", "tumor_id", "normal_id", "tool", "chrom", "sv_event_involvement_count"],
    )
    write_table(
        tables_dir / "trans_chromosome_pair_counts.by_pair_tool.tsv",
        trans_chr_pair_distribution(all_records),
        ["pair_id", "tumor_id", "normal_id", "tool", "chrom1", "chrom2", "svtype", "count"],
    )
    write_table(
        tables_dir / "svtype_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "svtype"]),
        ["tool", "svtype", "count"],
    )
    write_table(
        tables_dir / "length_bin_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "length_bin"]),
        ["tool", "length_bin", "count"],
    )
    write_table(
        tables_dir / "cis_trans_counts.overall_by_tool.tsv",
        count_rows(all_records, ["tool", "relation"]),
        ["tool", "relation", "count"],
    )

    print(f"[DONE] records={len(all_records)}")
    print(f"[TABLES] {tables_dir}")
    print("[PLOTS] skipped")


if __name__ == "__main__":
    main()


## 二、运行

### 运行所需的 80_pairs.six_sv_tools.manifest.new.tsv 文件见 draw_upset.ipynb

In [ ]:
python summarize_80_pairs_sv_by_tool.py \
  --manifest /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv \
  --outdir /mnt/home/ygjx/chenkejin/80_sv_summary